# Landslide / SAR change segmentation

**Kaggle settings (right sidebar):** Accelerator → **GPU T4 x2** (or P100) · Internet → **On** · Persistence → Files (optional).
Optional: add a Kaggle Secret named `HF_TOKEN` (Add-ons → Secrets) to push models to Hugging Face or use gated datasets.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
# one GPU is enough; avoids multi-GPU DataParallel surprises
!nvidia-smi

In [ ]:
!git clone -q https://github.com/Sidvortex/sar-landslide-seg.git || (cd sar-landslide-seg && git pull -q)
%cd sar-landslide-seg
!pip install -q -U -r requirements.txt

In [ ]:
# optional: log in to Hugging Face with a Kaggle secret
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception as e:
    print('No HF_TOKEN secret, continuing anonymously:', type(e).__name__)

In [ ]:
!apt-get -qq install -y libgdal-dev > /dev/null 2>&1 || true  # rasterio wheels usually bundle GDAL; this is a fallback

## 1. Train on Landslide4Sense (downloads ~3 GB on first run, ≈1 h for 40 epochs on a T4)

In [ ]:
!python train.py --config configs/landslide4sense.yaml

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('outputs/l4s-unet-r34/history.json'))
plt.plot([r['epoch'] for r in h], [r['val_iou'] for r in h], label='val IoU')
plt.plot([r['epoch'] for r in h], [r['val_f1'] for r in h], label='val F1')
plt.legend(); plt.xlabel('epoch'); plt.show()

## 2. Try a stronger encoder (optional)

In [ ]:
!python train.py --config configs/landslide4sense.yaml --set model.encoder=efficientnet-b3 train.output_dir=outputs/l4s-effb3

## 3. Predict on a validation patch

In [ ]:
!python predict.py --ckpt outputs/l4s-unet-r34/best.pt --h5 data/landslide4sense/images/validation/image_1.h5 --out pred_1.png
from IPython.display import Image; Image('pred_1.png')

## 4. Your own SAR pairs
Upload a Kaggle dataset with `train/<scene>/pre.tif, post.tif, mask.tif` (see README) and point `data.root` at it:

In [ ]:
# !python train.py --config configs/sar_change_pairs.yaml --set data.root=/kaggle/input/<your-dataset>

In [ ]:
!cd outputs && zip -qr /kaggle/working/sar_models.zip .
!ls -lh /kaggle/working/*.zip